# 03 · Contrastive refinement (DDP, 2 GPUs)
Each aligned student (28, 14, 7 layers) is trained further on (query, positive, negative) triples so it ranks
the right passage first, not only lands near the teacher query vector.

The **document side is frozen**: documents are the teacher's `corpus.npy` vectors from `01`, never re-encoded.
Only the student's query encoder trains: every weight + head, since all three students come from the full
fine-tune `02` (`full_l28`, then pruned `prog_l14`, `prog_l7`).

Loss: the Qwen3-Embedding objective (arXiv 2506.05176, Eq. 1). Per GPU, for its B queries of a global batch of
G = B · NPROCS (student queries all-gathered with grad; positives are frozen teacher vectors, indexed on every GPU):
```
logits_i = [s(q_i, d_j+) for the G positives] ++ [s(q_i, d_ik-) for q_i's own n_neg negatives]
        ++ [s(q_i, q_j)  for the G queries  ] ++ [s(d_i+, d_j+) for the G positives]               / tau
masked   = q_i's labelled positives (incl. a shared positive), q_i itself, any negative with s > s(q_i, d_i+) + margin
L        = CE(logits, own positive) + l2_weight * ||q_i - teacher(q_i)||^2
```
Per query 3 · (G − 1) + n_neg negatives. The d-d term is teacher–teacher: no gradient, a constant in the denominator.

Diagnostics in every train row, plotted per student in section 3:
- `acc`: share of queries whose positive is the top logit.
- `s_pos`: mean s(q, d+).
- `gap`: s_pos minus the hardest unmasked negative.
- `m_<block>`: the share of that block's negatives that is masked.
- `z_<block>`: that block's share of the negative mass in the denominator (they sum to 1).
- `grad_norm`: gradient norm before clipping.

Masks move with the student, so read `cr` together with `m_*`.

Input: `01` (data + teacher vectors), the full-fine-tune `02` notebook output (students + metrics). Output (attach to `04`): `runs/<mode>/<student>_cr/`, `runs/<mode>/metrics.json`.

In [ ]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
secret_value_0 = user_secrets.get_secret("HF_TOKEN")


In [ ]:
# setup: clone/pull the repo on Kaggle, put it on sys.path, install what's missing (re-run to pull latest code)
import os, sys, importlib.util
REPO_URL = "https://github.com/pyther-hub/hlm-poc.git"
ON_KAGGLE = os.path.exists("/kaggle/working")
REPO = "/kaggle/working/hlm-poc" if ON_KAGGLE else os.path.abspath("..")
if ON_KAGGLE:
    if not os.path.exists(REPO):
        !git clone -q {REPO_URL} {REPO}
    else:
        !git -C {REPO} pull -q
sys.path.insert(0, REPO)

# HF token (higher rate limits, faster downloads). Kaggle: Add-ons -> Secrets -> add "HF_TOKEN", attach to notebook.
# Spawned workers inherit os.environ. Locally: export HF_TOKEN or `huggingface-cli login`.
if ON_KAGGLE and "HF_TOKEN" not in os.environ:
    try:
        from kaggle_secrets import UserSecretsClient
        os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    except Exception as e:
        print(f"HF_TOKEN secret not found ({type(e).__name__}): downloads stay unauthenticated")

import transformers
from packaging.version import Version
if Version(transformers.__version__) < Version("4.56"):  # Qwen3 + the dtype= kwarg
    !pip install -q -U "transformers>=4.56"
    print("transformers upgraded: restart the kernel, then re-run this cell")
for pkg in ("datasets", "pyarrow", "accelerate"):
    if importlib.util.find_spec(pkg) is None:
        !pip install -q {pkg}

# %load_ext autoreload
# %autoreload 2

## Control panel

In [ ]:
import os
import shutil
import torch
from model import get_device
from teacher import TEACHER_DEFAULTS as T

DEBUG = False    
NPROCS = 2
N_NEG = 1        # experiment: 1 (stored negative[0] only) vs 8 (+ 7 teacher-mined); each query sees only its own

# --- paths. On Kaggle, earlier notebooks' outputs are attached as inputs (read only); set the slugs to yours.
#     Off Kaggle everything lives under the repo, so every IN_* is the same dir.
ROOT = "/kaggle/working" if ON_KAGGLE else REPO
MODE = "debug" if DEBUG else "full"
KAGGLE_IN = "/kaggle/input/datasets/tensorpanda231"


IN_01 = f"{KAGGLE_IN}/corpus-teacher-embd" if ON_KAGGLE else REPO
# 02 = the full-fine-tune run's outputs (full_l28, prog_l14, prog_l7); check the path with Kaggle's "Copy file path"
IN_02 = f"{KAGGLE_IN}/fft-qwen-emb-models" if ON_KAGGLE else REPO
DATA_PATH = f"{IN_01}/cache/{MODE}/data.json"
EMB_DIR = f"{IN_01}/cache/{MODE}/emb/teacher"
RUN_DIR = f"{ROOT}/runs/{MODE}"
PREV_METRICS = f"{IN_02}/runs/{MODE}/metrics.json"

# --- students to refine: name -> checkpoint dir from 02 (row "<name>_cr" each)
STUDENTS = {n: f"{IN_02}/runs/{MODE}/{n}/final" for n in ("full_l28", "prog_l14", "prog_l7")}

# --- batch per student (manual). Queries per GPU per step = batch_size * grad_accum: keep it 96 so the lr holds.
#     In-batch negatives come from one micro-batch: per query 3 * (batch_size * NPROCS - 1) + n_neg negatives.
BATCH = dict(full_l28=dict(batch_size=16, grad_accum=6),   # 596M, every weight trains (embeddings too): 16 peaked 13.4/14.6 GB
             prog_l14=dict(batch_size=96, grad_accum=1),
             prog_l7=dict(batch_size=96, grad_accum=1))

# --- refinement (refine.REFINE_DEFAULTS; init_adapter / out_dir set per student)
REFINE = dict(dim=T["dim"], query_prefix="", max_query_len=32,   # same as 02
              tau=0.05,                            # temperature (Qwen3 gives no value)
              margin=0.1,                          # mask negatives with s > s(q, d+) + margin (Qwen3 Eq. 1: 0.1)
              n_neg=N_NEG,                         # own hard negatives per query: 1 = stored one, k > 1 adds k-1 mined
              l2_weight=0.1,                       # keeps the student near the teacher query space; 0 = pure Qwen3
                                                   # (old loss at 0: val_cos fell 0.88 -> 0.75, nDCG@10 -0.04-0.05)
              lr=1e-5,                             # every weight trains; 5e-5 drifted off the teacher in 200 steps
              head_lr=1e-4, weight_decay=0.0, warmup_ratio=0.05,
              epochs=1,                           # batch_size / grad_accum: BATCH
              grad_clip=1.0, amp=True, seed=0,
              log_every=20, eval_every=50,         # ~470 steps / epoch -> ~9 evals (~20 s each)
              patience=3,                          # evals without a stop_on gain -> stop; best weights -> final/
              stop_on="val_ndcg@10",               # not val_loss (= L2 to the teacher, rises under CR)
              save_every=0,                        # weights only: a full student's step checkpoint is ~7 GB with AdamW
              keep_last=2,
              data_path=DATA_PATH, teacher_emb_dir=EMB_DIR)

# --- evaluation (evaluation.evaluate_model kwargs): student queries vs the frozen teacher corpus, top 100
EVAL = dict(splits=("val", "test"), dim=T["dim"], max_query_len=32, query_prefix="", device=get_device(), dtype="fp16")

if DEBUG:
    REFINE.update(log_every=5, eval_every=20)

## 1 · Check inputs

In [ ]:
from data import load_json
from evaluation import load_index

# rows accumulate across notebooks: start from the previous notebook's metrics.json (copied once)
os.makedirs(RUN_DIR, exist_ok=True)
if not os.path.exists(f"{RUN_DIR}/metrics.json") and os.path.exists(PREV_METRICS):
    shutil.copy(PREV_METRICS, f"{RUN_DIR}/metrics.json")
data = load_json(DATA_PATH)
index = load_index(data, EMB_DIR, ("val", "test"))
assert set(data["train"]["ids"]) <= set(data["align"]["ids"]), "train queries need teacher vectors (align set)"
for name, path in STUDENTS.items():
    cfg = load_json(f"{path}/adapter_config.json")
    assert cfg["out_dim"] == T["dim"]
    print(f"{name:<9} layers {len(cfg.get('keep_layers', range(28)))}  full fine-tune: {cfg.get('full', False)}")
n_train = len(data["train"]["ids"])
print(f"{n_train} train queries")
for name, b in BATCH.items():
    G = b["batch_size"] * NPROCS  # global batch of one micro-step
    print(f"{name:<9} {n_train // (G * b['grad_accum'])} steps, global batch {G} queries, "
          f"{3 * (G - 1) + REFINE['n_neg']} negatives per query (n_neg {REFINE['n_neg']})")

## 2 · The training triples
Each train query comes with its labelled positive(s) and the stored negative (`negative_passages[0]`, not verified BM25).

In [ ]:
corpus = data["corpus"]["texts"]
for i in range(3):
    print(f"query    {data['train']['texts'][i]}")
    print(f"positive {corpus[data['train']['pos'][i][0]][:200]}")
    print(f"negative {corpus[data['train']['neg'][i]][:200]}\n")
    

## 3 · Refine every student, compare with its pre-refinement checkpoint
Trains the missing `<name>_cr` arms (batch from `BATCH`) and plots each one's curves, then evaluates the missing
rows and prints the gain per student and the table.

In [ ]:
import matplotlib.pyplot as plt
from IPython.display import display
from refine import refine_main
from run_exp import plot_curves, save_fig, train_once

def show_run(name):
    """One run, one panel per log key: training curves, in-batch health, each block's share of the denominator,
    masked share per block; then disk use."""
    panels = dict(curves=("lr", "train_loss", "cr", "val_loss", "val_ndcg@10"),
                  health=("acc", "s_pos", "gap", "grad_norm", "l2"),
                  z=("z_qd", "z_neg", "z_qq", "z_dd"),
                  masked=("m_qd", "m_neg", "m_qq", "m_dd"))
    for tag, keys in panels.items():
        fig = plot_curves({name: f"{RUN_DIR}/{name}"}, keys, f"{name} · {tag}")
        display(save_fig(fig, RUN_DIR, f"{name}_{tag}"))
        plt.close("all")
    !du -sh {RUN_DIR}
    !df -h {ROOT}

for name, init in STUDENTS.items():
    cfg = {**REFINE, **BATCH[name], "init_adapter": init, "out_dir": f"{RUN_DIR}/{name}_cr"}
    train_once(refine_main, cfg, NPROCS)
    show_run(f"{name}_cr")

In [ ]:
from run_exp import exp_cr

# every <name>_cr/final exists now, so exp_cr only evaluates, prints the CR gains and plots
res = exp_cr(REFINE, STUDENTS, data, index, RUN_DIR, EVAL, NPROCS)
for fig in res["figs"]:
    display(fig)
plt.close("all")